In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_github_docs(url):
    # Set a user-agent to mimic a browser
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    print(f'Fetching: {url}')
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f'Failed to retrieve page: {response.status_code}')
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

# Target URL
url = 'https://docs.github.com/en'
soup = scrape_github_docs(url)

Fetching: https://docs.github.com/en


In [4]:
categories = []

# GitHub Docs often uses 'h2' or specific class names for product cards
# We'll look for common patterns in their HTML structure
for card in soup.find_all('div', class_='Box-sc-1gh2r6s-0'): # Note: GitHub's classes may change
    title_elem = card.find('h3')
    desc_elem = card.find('p')

    if title_elem:
        categories.append({
            'Title': title_elem.get_text(strip=True),
            'Description': desc_elem.get_text(strip=True) if desc_elem else 'No description'
        })

# Create a DataFrame to display the results
df = pd.DataFrame(categories)
display(df)

""


In [5]:
def scrape_commands(product_url):
    soup = scrape_github_docs(product_url)
    if not soup:
        return []

    commands = []
    # GitHub often uses article sections or specific div classes for command references
    # This logic targets the structure found in many of their CLI and API reference pages
    for section in soup.find_all(['div', 'section'], class_='gh-command'): # Example class
        name = section.find(['h2', 'h3'])
        usage = section.find('code')
        description = section.find('p')

        if name:
            commands.append({
                'Command': name.get_text(strip=True),
                'Usage': usage.get_text(strip=True) if usage else 'N/A',
                'Details': description.get_text(strip=True) if description else 'N/A'
            })

    # If the specific class isn't found, try a generic approach for heading-based structures
    if not commands:
        for header in soup.find_all('h2'):
            next_p = header.find_next_sibling('p')
            if next_p:
                commands.append({
                    'Command': header.get_text(strip=True),
                    'Details': next_p.get_text(strip=True)
                })

    return commands

# Example: Scraping GitHub CLI commands page
cli_url = 'https://cli.github.com/manual/' # CLI manual is a great source for commands
cli_commands = scrape_commands(cli_url)

command_df = pd.DataFrame(cli_commands)
display(command_df.head(20))

Fetching: https://cli.github.com/manual/


,Command,Details
0,Installation,You can find installation instructions on ourR...
1,Configuration,GitHub CLI supports GitHub Enterprise Server 2...
2,GitHub Enterprise,GitHub CLI supports GitHub Enterprise Server 2...


In [6]:
def scrape_git_scm_docs(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    git_commands = []

    # Git-SCM documentation lists commands within 'ul.unstyled' inside the reference sections
    # Each list item contains an 'a' for the command and a 'span' for the description
    for li in soup.select('ul.unstyled li'):
        link = li.find('a')
        summary = li.find('span', class_='description')

        if link:
            git_commands.append({
                'Command': link.get_text(strip=True),
                'Link': 'https://git-scm.com' + link['href'],
                'Summary': summary.get_text(strip=True) if summary else 'N/A'
            })

    return git_commands

git_url = 'https://git-scm.com/docs'
git_data = scrape_git_scm_docs(git_url)

git_df = pd.DataFrame(git_data)
display(git_df.head(20))

,Command,Link,Summary
0,git,https://git-scm.com/docs/git,N/A
1,config,https://git-scm.com/docs/git-config,N/A
2,help,https://git-scm.com/docs/git-help,N/A
3,bugreport,https://git-scm.com/docs/git-bugreport,N/A
4,Credential helpers,https://git-scm.com/doc/credential-helpers,N/A
5,init,https://git-scm.com/docs/git-init,N/A
6,clone,https://git-scm.com/docs/git-clone,N/A
7,add,https://git-scm.com/docs/git-add,N/A
8,status,https://git-scm.com/docs/git-status,N/A
9,diff,https://git-scm.com/docs/git-diff,N/A


In [7]:
# Scrape all commands (increased limit)
if 'git_df' in globals():
    # Using 999 as a high limit to capture all available commands
    full_command_details = scrape_command_details(git_df, limit=999)

    # Save to JSON file
    output_file = 'git_commands_documentation.json'
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(full_command_details, f, indent=4)

    print(f'\nSuccessfully saved documentation for {len(full_command_details)} commands to {output_file}')
else:
    print("Error: 'git_df' not found. Please run the previous cell (1dbe5d02) first.")

NameError: name 'scrape_command_details' is not defined

In [ ]:
import os

# Define the directory in Google Drive
drive_path = '/content/drive/My Drive/rag_git/git_scraper_results'
os.makedirs(drive_path, exist_ok=True)

# Update the output file path
drive_output_file = os.path.join(drive_path, 'git_commands_documentation.json')

if 'full_command_details' in globals():
    with open(drive_output_file, 'w', encoding='utf-8') as f:
        json.dump(full_command_details, f, indent=4)
    print(f'Successfully saved documentation to Google Drive at: {drive_output_file}')
else:
    print('Error: full_command_details not found. Please run the scraping cell first.')

In [ ]:
import datetime
import re
import json
import os
import pandas as pd

def clean_text(text):
    # Remove multiple newlines and spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def prepare_for_rag_v2(details_list):
    rag_ready_data = []

    for entry in details_list:
        command_name = entry['command']
        content = entry['full_details']
        url = entry['url']

        # 1. Section-based chunking (Basic Logic)
        # Since the scraper currently gets a large block, we identify sections by keyword patterns
        # In a production scenario, we'd parse the HTML structure more granularly.
        sections = {
            "description": re.split(r'OPTIONS|EXAMPLES|SYNOPSIS', content, flags=re.IGNORECASE)[0],
            "usage": "".join(re.findall(r'SYNOPSIS(.*?)(?:DESCRIPTION|OPTIONS|EXAMPLES|$)', content, flags=re.DOTALL | re.IGNORECASE))
        }

        for section_name, text in sections.items():
            if text.strip() and len(text.strip()) > 20:
                rag_ready_data.append({
                    "text": f"Command: {command_name}. Section: {section_name}. Content: {clean_text(text)}",
                    "command": command_name,
                    "section": section_name,
                    "option": None,
                    "source": "git_docs",
                    "metadata": {"url": url, "scraped_at": datetime.datetime.now().isoformat()}
                })

        # 2. Option-level chunking
        # Extract patterns like --option or -o followed by text
        options_section = "".join(re.findall(r'OPTIONS(.*?)(?:EXAMPLES|DESCRIPTION|$)', content, flags=re.DOTALL | re.IGNORECASE))
        option_matches = re.findall(r'(--[a-z0-9\-]+)(.*?)(?=--[a-z0-9\-]|\Z)', options_section, flags=re.DOTALL)

        for opt_name, opt_desc in option_matches:
            rag_ready_data.append({
                "text": f"Command {command_name} option {opt_name}: {clean_text(opt_desc)}",
                "command": command_name,
                "section": "option",
                "option": opt_name,
                "source": "git_docs",
                "metadata": {"url": url}
            })

        # 3. Example-level chunking
        examples_section = "".join(re.findall(r'EXAMPLES(.*?)(?:$)', content, flags=re.DOTALL | re.IGNORECASE))
        if examples_section.strip():
            rag_ready_data.append({
                "text": f"Example for {command_name}: {clean_text(examples_section)}",
                "command": command_name,
                "section": "example",
                "option": None,
                "source": "git_docs",
                "metadata": {"url": url}
            })

    return rag_ready_data

if 'full_command_details' in globals():
    rag_data = prepare_for_rag_v2(full_command_details)

    rag_output_file = os.path.join(drive_path, 'git_rag_refined_ingestion.jsonl')
    with open(rag_output_file, 'w', encoding='utf-8') as f:
        for item in rag_data:
            f.write(json.dumps(item) + '\n')

    print(f'Refined RAG data saved to: {rag_output_file}')
    display(pd.DataFrame(rag_data).head())
else:
    print('Please run the scraping cells first.')